In [ ]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from IPython.display import Image
from io import BytesIO
#from scrapy import fetch_website_links, fetch_website_contents
from openai import OpenAI

import requests
from bs4 import BeautifulSoup

import pandas as pd
import ast   # or json if it's valid JSON
from urllib.parse import urljoin

from parsing_url import redacted_links, redacted_contents, \
    redacted_images, redacted_headings, redacted_metadata

## Select Models

In [ ]:
# Initialize and constants
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")    

openai = OpenAI()

In [ ]:
def system_prompt_fetch_data_from_url():
    SYSTEM_PROMPT_FETCH_DATA = ''' 
    You are an expert biographer and researcher. 
Your task is to collect and summarize content from the given URL about a specific person. 
You must determine if the content is relevant to the person or not. 
Relevant content can describe achievements, controversies, family, or other aspects of their life.
Focus on main areas:

A. Achievements (categorize into fields)
1. Science & Technology - discoveries, inventions, computing, AI, space
2. Arts & Literature - painting, music, theater, cinema, writing, philosophy
3. Politics & Leadership - governance, reforms, diplomacy, military vision
4. Culture & Society - traditions, education, social welfare, identity
5. Economics & Business - entrepreneurship, trade, philanthropy
6. Sports & Physical Achievement - athletic excellence, teamwork, national pride
7. Humanitarian & Spiritual - religious leadership, charity, peacebuilding
8. War Heroes - military service, sacrifice, defense of nation

B. Personal Characteristics
1. Identity & Background - name, birth, family, education
2. Personality Traits - strengths, weaknesses, influence on decisions
3. Values & Beliefs - moral compass, religious/philosophical/political views
4. Professional Characteristics - dedication, loyalty, contributions, leadership
5. Social & Family - relationships, family sentiment, influence on career
6. Controversies & Challenges - scandals, criticisms, wrong decisions, responses
7. Impact & Legacy - influence on society, institutions inspired, remembrance

Constraints:
- Fetched content must not exceed 500 words.
- Output must be in JSON format:
{
  "Content relations with person": "Content"
}
    '''
    return SYSTEM_PROMPT_FETCH_DATA

In [ ]:
def user_prompt_fetch_data(name: str, url: str,content: str) -> str:
    """
    Build a user prompt that includes fetched content from the URL.
    """  
    
    USER_PROMPT_FETCH_DATA = f"""
You are an expert biographer and researcher. 
Examine if the {content} fetched from {url} is about {name}. 
If relevant, create structured content covering achievements, identity, traits, values, professional life, family, controversies, and legacy. 
Fetched content (limited to 500 words):

{content}

Output must be JSON:
{{
    "Content relations with person": "{content}"
}}
"""
    return USER_PROMPT_FETCH_DATA

In [ ]:
def get_list_of_url(url: str) -> list:
    headers = {"User-Agent": "Mozilla/5.0"}
    # Assuming redacted_links returns a list of URLs
    links = redacted_links(url, headers)
    # Normalize links
    normalized_links = [urljoin(url, link) for link in links]
    # Return the main URL plus normalized links
    return [url] + normalized_links

In [ ]:
class ContentFetchError(Exception):
    """Raised when link fetching fails."""
    pass

class ModelCallError(Exception):
    """Raised when OpenAI model call fails."""
    pass

def get_relevant_contents_from_models(person_name: str,
                                      main_url: str,
                                      model: str = "gpt-4o-mini",
                                      max_num_of_links_for_examin: int = 3) -> list:
    """
    Fetch relevant contents about a person from all links related to the main URL using the model.
    """
    results = []
    headers = {"User-Agent": "Mozilla/5.0"}

    # Step 1: Fetch links
    try:
        links = get_list_of_url(main_url)
        if not links:
            raise ContentFetchError(f"No links found for {main_url}")
        links = links[:max_num_of_links_for_examin]
        print(f"[Info] Found {len(links)} links to examine.")
    except Exception as e:
        print(f"[Error] Failed to fetch links from {main_url}: {e}")
        return []

    # Step 2: Iterate through links
    for link in links:
        try:
            content = redacted_contents(link, headers)

            # Normalize content if it's a list
            if isinstance(content, list):
                content = " ".join(str(c) for c in content if c)

            if not content or (isinstance(content, str) and content.strip() == ""):
                print(f"[Warning] Empty content at {link}")
                continue
        except Exception as e:
            print(f"[Error] Failed to fetch content from {link}: {e}")
            continue

        # Step 3: Call the model
        try:
            user_prompt = user_prompt_fetch_data(person_name, link, content)
            response = openai.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt_fetch_data_from_url()},
                    {"role": "user", "content": user_prompt}
                ],
                response_format={"type": "json_object"}
            )
            result = response.choices[0].message.content
            print(f"[Info] Model returned content for {link}")
            results.append(result)
        except Exception as e:
            print(f"[Error] Model call failed for {link}: {e}")
            raise ModelCallError(f"Model call failed for {link}") from e

    # Step 4: Final summary
    if not results:
        print(f"[Warning] No relevant results generated for {person_name}.")
    else:
        print(f"[Info] Generated {len(results)} relevant results for {person_name}.")

    return results


In [ ]:

'''person_info = get_relevant_contents_from_models(person_name = person,
                                      main_url = murl,
                                      model = MODEL,
                                      max_num_of_links_for_examin = num_examin_link)
person_info'''

In [ ]:
def get_bigraphy_human_sytem_prompt():
    SYSTEM_PROMPT_BIOGRAPHY = f'''
    You are an expert biographer writer.
    Your task is to summarize given content about a specific person and write biography.
    Output must be in Biography format as a valid JSON object.
    '''
    return SYSTEM_PROMPT_BIOGRAPHY

def get_bigraphy_human_user_prompt(name : str, contents : str):
    BIOGRAPHY_USER_PROMPT = f'''
    Write biography of person {name} from given {contents}.
    Output must be in Biography format as valid JSON.
    '''
    return BIOGRAPHY_USER_PROMPT

In [ ]:
def clean_contents(contents: list) -> str:
    """Filter out irrelevant entries and merge into one string."""
    relevant = []
    for c in contents:
        if isinstance(c, str) and "Not relevant" not in c:
            relevant.append(c)
    return " ".join(relevant)

In [ ]:
class EmptyContentsError(Exception):
    """Raised when get_relevant_contents_from_models returns empty content."""
    pass

def genrate_biography(person_name: str, main_url: str,
                      model: str = "gpt-4o-mini",
                      max_num_of_links_for_examin: int = 3):

    contents = get_relevant_contents_from_models(
        person_name=person_name,
        main_url=main_url,
        model=model,
        max_num_of_links_for_examin=max_num_of_links_for_examin
    )   
        
    # ✅ Fix: contents is a list, not a string
    if not contents or len(contents) == 0:
        print(f"[Warning] No relevant contents found for {person_name} from {main_url}")
        raise EmptyContentsError(
            f"No relevant contents found for {person_name} from {main_url}"
        )
    else:
        # ✅ Merge list items into one string for the model
        merged_contents = clean_contents(contents)
        if not merged_contents.strip():
            print(f"[Warning] All contents were irrelevant for {person_name}")
            raise EmptyContentsError("No relevant biography data found.")   
    # ✅ Write merged contents to file in current working directory
    file_name = f"{person_name}_{model}_biography.txt"
    file_path = os.path.abspath(file_name)

    # Open in write mode (this automatically cleans/truncates if file exists)
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(merged_contents)

    print(f"[Info] Merged contents written to {file_path}")
    # Call the model
    response = openai.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": get_bigraphy_human_sytem_prompt()},
            {"role": "user", "content": get_bigraphy_human_user_prompt(name=person_name, contents=merged_contents)}
        ],
        response_format={"type": "json_object"},
        max_completion_tokens=2000,
        n=1,
        user="abhishek"
    )

    result = response.choices[0].message.content
    return result


In [ ]:
person = "Albert Einstein"
murl = "https://en.wikipedia.org/wiki/Albert_Einstein"
MODEL = 'gpt-4o-mini'
num_examin_link = 3

In [ ]:
BIOGRAPHY = genrate_biography(person_name = person, 
                  main_url = murl, 
                  model = MODEL,
                  max_num_of_links_for_examin=num_examin_link)

In [ ]:
def get_text_from_json_object(json_object: dict) -> str:
    """
    Recursively convert a JSON object (dict) into a human-readable text format.
    Handles nested dicts and lists automatically.
    """
    def recurse(obj, indent=0):
        lines = []
        prefix = "  " * indent
        if isinstance(obj, dict):
            for key, value in obj.items():
                if isinstance(value, (dict, list)):
                    lines.append(f"{prefix}{key.replace('_',' ').capitalize()}:")
                    lines.extend(recurse(value, indent + 1))
                else:
                    lines.append(f"{prefix}{key.replace('_',' ').capitalize()}: {value}")
        elif isinstance(obj, list):
            for item in obj:
                if isinstance(item, (dict, list)):
                    lines.extend(recurse(item, indent + 1))
                else:
                    lines.append(f"{prefix}- {item}")
        else:
            lines.append(f"{prefix}{obj}")
        return lines

    return "\n".join(recurse(json_object))

In [ ]:
bio_data = json.loads(BIOGRAPHY)
text_output = get_text_from_json_object(bio_data)
print(text_output)

#### Work With Olama

In [31]:
import requests

response = requests.get("http://localhost:11434/api/tags")
if response.status_code == 200:
    models = response.json()
    print("Available Ollama models:")
    for model in models.get("models", []):
        print(f"- {model['name']}")
else:
    print("Error fetching models:", response.text)


Available Ollama models:
- llama3.2:latest
